## Etapa 4. Preparação dos dados e modelagem

Agora vamos preparar os dados para o treinamento dos modelos de Machine Learning.

### 4.1. Importação das bibliotecas

In [1]:
# Importação das bibliotecas

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error

### 4.2. Carregamento dos dados

In [2]:
from pathlib import Path

# Caminho para a pasta principal do projeto
PROJECT_DIR = Path.cwd().parent

# Carregamento dos dados
train = pd.read_csv(
    PROJECT_DIR / "train.csv",
    nrows=188318
)

test = pd.read_csv(
    PROJECT_DIR / "test.csv",
    nrows=125546
)

# Conferência das dimensões
print("Treino:", train.shape)
print("Teste:", test.shape)

Treino: (188318, 132)
Teste: (125546, 131)


### 4.3. Separação das variáveis explicativas e da variável-alvo

- X: variáveis que o modelo utilizará para fazer previsões.
- y: variável que queremos prever, ou seja, loss.

In [3]:
# Separação das variáveis explicativas e da variável-alvo

X = train.drop(columns=["id", "loss"])
y = train["loss"]

# Conferência
print("Dimensões de X:", X.shape)
print("Dimensões de y:", y.shape)

Dimensões de X: (188318, 130)
Dimensões de y: (188318,)


**Divisão dos dados em treino e validação**

Nesta etapa, dividiremos o conjunto de treino em duas partes: 80% para treinar o modelo e 20% para validar seu desempenho. Essa separação permite avaliar a capacidade de previsão do modelo utilizando dados que não foram apresentados durante o treinamento.

### 4.4. Divisão dos dados em treino e validação

In [4]:
# Divisão dos dados em treino e validação

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Conferência das dimensões
print("Treino:", X_train.shape)
print("Validação:", X_valid.shape)

Treino: (150654, 130)
Validação: (37664, 130)


### 4.5. Identificação das variáveis categóricas para o CatBoost

O CatBoost consegue trabalhar diretamente com variáveis categóricas, sem precisarmos transformá-las previamente em números usando técnicas como One-Hot Encoding.

In [5]:
# Identificar as colunas categóricas

categorical_features = X_train.select_dtypes(
    include=["str"]
).columns.tolist()

print("Quantidade de variáveis categóricas:", len(categorical_features))

Quantidade de variáveis categóricas: 116


### 4.7. Importação dos modelos


In [6]:
# Importação dos algoritmos de Machine Learning

from catboost import CatBoostRegressor
from sklearn.ensemble import RandomForestRegressor

### 4.8. Definição das métricas de avaliação
Precisamos escolher como vamos medir os erros das previsões dos modelos.

Vamos utilizar duas métricas:
- MAE (Mean Absolute Error): calcula o erro absoluto médio entre os valores previstos e os valores reais. Quanto menor, melhor.
- RMSE (Root Mean Squared Error): calcula a raiz do erro quadrático médio. Penaliza mais fortemente os erros muito grandes.

In [7]:
# Importação das métricas de avaliação

from sklearn.metrics import mean_absolute_error, root_mean_squared_error

### 4.9. Configuração do CatBoost

In [8]:
# Configuração do modelo CatBoost

catboost_model = CatBoostRegressor(
    iterations=500,
    learning_rate=0.05,
    depth=8,
    loss_function="RMSE",
    random_seed=42,
    verbose=100
)

### 4.10. Treinamento do CatBoost

Nesta etapa, o modelo vai aprender a identificar padrões entre as 130 variáveis explicativas e os valores de loss.

In [10]:
# Treinamento do modelo CatBoost

catboost_model.fit(
    X_train,
    y_train,
    cat_features=categorical_features,
    eval_set=(X_valid, y_valid),
    early_stopping_rounds=50
)

0:	learn: 2855.9320715	test: 2798.4109033	best: 2798.4109033 (0)	total: 3.65s	remaining: 30m 20s
100:	learn: 1906.4936827	test: 1921.2531407	best: 1921.2531407 (100)	total: 5m 23s	remaining: 21m 18s
200:	learn: 1839.4526032	test: 1878.6248307	best: 1878.6248307 (200)	total: 13m 17s	remaining: 19m 46s
300:	learn: 1799.0212921	test: 1862.2559145	best: 1862.2559145 (300)	total: 19m 25s	remaining: 12m 50s
400:	learn: 1775.0497518	test: 1855.8462423	best: 1855.7937598 (398)	total: 24m 45s	remaining: 6m 6s
499:	learn: 1758.4054710	test: 1852.2634810	best: 1852.2634810 (499)	total: 29m 57s	remaining: 0us

bestTest = 1852.263481
bestIteration = 499



CatBoostRegressor(depth=8, iterations=500, learning_rate=0.05, loss_function='RMSE', random_seed=42, verbose=100)

### 4.11. Gerar previsões com o CatBoost
Vamos utilizar o modelo treinado para prever os valores de loss do conjunto de validação.

In [11]:
# Gerar previsões para os dados de validação

y_pred_catboost = catboost_model.predict(X_valid)

# Visualizar as primeiras previsões
print(y_pred_catboost[:10])

[1531.27313746 1406.77436493 3183.18235276 2700.28672923 1359.81575111
 4045.4820005  3152.84936958 1936.92618823 2405.89130165 1343.34465639]


### 4.12. Avaliar o desempenho do CatBoost
Depois vamos calcular os erros

In [12]:
# Avaliação do modelo CatBoost

mae_catboost = mean_absolute_error(y_valid, y_pred_catboost)
rmse_catboost = root_mean_squared_error(y_valid, y_pred_catboost)

print("Resultados do CatBoost:")
print("MAE:", round(mae_catboost, 2))
print("RMSE:", round(rmse_catboost, 2))

Resultados do CatBoost:
MAE: 1176.09
RMSE: 1852.26


### 4.13. Preparar as variáveis categóricas
O CatBoost consegue trabalhar diretamente com categorias em formato de texto. Já o Random Forest precisa receber valores numéricos.

In [13]:
from sklearn.preprocessing import OrdinalEncoder

### 4.14. Codificar as variáveis categóricas
Vamos criar cópias dos dados de treino e validação para o Random Forest. Assim, preservamos os dados originais, que continuam sendo utilizados pelo CatBoost.

In [14]:
# Criar cópias dos conjuntos de dados
X_train_rf = X_train.copy()
X_valid_rf = X_valid.copy()

# Configurar o codificador
encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

# Aprender as categorias no treino e transformar os dois conjuntos
X_train_rf[categorical_features] = encoder.fit_transform(
    X_train_rf[categorical_features]
)

X_valid_rf[categorical_features] = encoder.transform(
    X_valid_rf[categorical_features]
)

print("Dimensões do treino:", X_train_rf.shape)
print("Dimensões da validação:", X_valid_rf.shape)

Dimensões do treino: (150654, 130)
Dimensões da validação: (37664, 130)


### 4.15. Configurar o modelo Random Forest
O Random Forest cria várias árvores de decisão e combina suas previsões. Essa combinação ajuda a reduzir a dependência de uma única árvore.

Vamos começar com 100 árvores e limitar a profundidade para controlar o tempo de treinamento.

In [15]:
# Configuração do modelo Random Forest

rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=20,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

### 4.16. Treinar o modelo Random Forest
O modelo vai aprender padrões entre as 130 variáveis de entrada e os valores de loss, utilizando os mesmos dados de treino usados pelo CatBoost.

In [16]:
# Treinamento do modelo Random Forest

rf_model.fit(
    X_train_rf,
    y_train
)

,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",20
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",2
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max

### 4.17. Gerar previsões com o Random Forest

In [17]:
# Gerar previsões para os dados de validação

y_pred_rf = rf_model.predict(X_valid_rf)

# Visualizar as primeiras previsões
print(y_pred_rf[:10])

[1886.52485876 1437.27019088 4158.36259941 2831.33027095 1251.99087232
 4691.0948512  2686.81631807 2144.01011058 1860.77514197 1477.43291925]


### 4.18. Avaliar o desempenho do Random Forest

In [18]:
# Avaliação do modelo Random Forest

mae_rf = mean_absolute_error(y_valid, y_pred_rf)
rmse_rf = root_mean_squared_error(y_valid, y_pred_rf)

print("Resultados do Random Forest:")
print("MAE:", round(mae_rf, 2))
print("RMSE:", round(rmse_rf, 2))

Resultados do Random Forest:
MAE: 1225.16
RMSE: 1902.49


### 4.19. Comparação dos modelos

In [19]:
# Comparação dos modelos

results = pd.DataFrame({
    "Modelo": ["CatBoost", "Random Forest"],
    "MAE": [mae_catboost, mae_rf],
    "RMSE": [rmse_catboost, rmse_rf]
})

results

,Modelo,MAE,RMSE
0,CatBoost,1176.090022,1852.263480
1,Random Forest,1225.164783,1902.488601


### 4.20. Conclusão da comparação dos modelos

### Conclusão da comparação dos modelos

Os dois algoritmos apresentaram resultados próximos, mas o CatBoost obteve o melhor desempenho nas duas métricas avaliadas.

O CatBoost alcançou MAE de 1.176,09 e RMSE de 1.852,26, enquanto o Random Forest apresentou MAE de 1.225,16 e RMSE de 1.902,49.

A diferença indica que o CatBoost conseguiu realizar previsões ligeiramente mais precisas para a variável `loss` no conjunto de validação.

Com base nesses resultados, o CatBoost foi selecionado para a etapa final de previsão, na qual será treinado novamente utilizando todos os registros disponíveis no conjunto original de treino.

### 4.21. Preparar os dados para o treinamento final
Agora precisamos fazer algo importante: até aqui, o CatBoost aprendeu com apenas 80% dos dados de treino, pois reservamos 20% para validação.

Para a previsão final, vamos aproveitar os 188.318 registros disponíveis.

In [9]:
# Preparar os dados completos para o treinamento final

X_full = train.drop(columns=["id", "loss"])
y_full = train["loss"]

print("Dimensões de X:", X_full.shape)
print("Dimensões de y:", y_full.shape)

Dimensões de X: (188318, 130)
Dimensões de y: (188318,)


### 4.22. Treinar o CatBoost com todos os dados

In [10]:
# Configurar o modelo final

final_model = CatBoostRegressor(
    iterations=500,
    learning_rate=0.05,
    depth=8,
    loss_function="RMSE",
    random_seed=42,
    verbose=100
)

In [11]:
# Treinamento final com todos os dados disponíveis

final_model.fit(
    X_full,
    y_full,
    cat_features=categorical_features
)

0:	learn: 2843.2007841	total: 4.62s	remaining: 38m 26s
100:	learn: 1897.3036633	total: 5m 17s	remaining: 20m 53s
200:	learn: 1836.2227040	total: 10m 25s	remaining: 15m 30s
300:	learn: 1803.0571012	total: 15m 33s	remaining: 10m 17s
400:	learn: 1781.9217608	total: 20m 48s	remaining: 5m 8s
499:	learn: 1765.8452675	total: 25m 56s	remaining: 0us


CatBoostRegressor(depth=8, iterations=500, learning_rate=0.05, loss_function='RMSE', random_seed=42, verbose=100)

### 4.22. Salvar o modelo final

Após o treinamento com os 188.318 registros, vamos salvar o modelo em um arquivo para que ele possa ser reutilizado sem precisar executar o treinamento novamente.

In [12]:
from pathlib import Path

# Criar a pasta para armazenar o modelo
models_dir = PROJECT_DIR / "models"
models_dir.mkdir(exist_ok=True)

# Salvar o modelo treinado
final_model.save_model(models_dir / "catboost_final.cbm")

### 4.23. Gerar previsões para o conjunto de teste

Agora utilizaremos o modelo treinado para prever os valores de loss dos 125.546 registros do conjunto de teste. Diferentemente do conjunto de treino, o arquivo de teste não possui os valores reais de loss.

In [17]:
# Remover a coluna identificadora
X_test = test.drop(columns=["id"])

# Gerar previsões
test_predictions = final_model.predict(X_test)

In [18]:
print("Registros no teste:", len(test))
print("Previsões geradas:", len(test_predictions))

Registros no teste: 125546
Previsões geradas: 125546


### 4.24. Criar o arquivo de submissão

O arquivo de submissão deve conter duas colunas: id, que identifica cada registro, e loss, que contém o valor previsto pelo modelo.

In [19]:
# Criar o arquivo de submissão
submission = pd.DataFrame({
    "id": test["id"],
    "loss": test_predictions
})

# Salvar o arquivo CSV
submission.to_csv(PROJECT_DIR / "submission.csv", index=False)

# Visualizar os primeiros registros
submission.head()

,id,loss
0,4,1783.291067
1,6,2025.737802
2,9,10149.278028
3,12,6560.047657
4,15,993.710577


### 4.25. Verificar o arquivo de submissão

In [20]:
# Conferir dimensões e estrutura do arquivo
print("Dimensões:", submission.shape)
print("\nColunas:", submission.columns.tolist())
print("\nValores ausentes:")
print(submission.isnull().sum())

# Conferir os valores previstos
print("\nEstatísticas das previsões:")
print(submission["loss"].describe())

Dimensões: (125546, 2)

Colunas: ['id', 'loss']

Valores ausentes:
id      0
loss    0
dtype: int64

Estatísticas das previsões:
count    125546.000000
mean       3017.686746
std        2123.527705
min         270.035781
25%        1641.626401
50%        2364.918337
75%        3649.201026
max       34759.369698
Name: loss, dtype: float64


### 4.26. Conferir o formato da submissão

In [21]:
# Carregar o arquivo de exemplo da competição
sample_submission = pd.read_csv(
    PROJECT_DIR / "sample_submission.csv",
    nrows=125546
)

# Comparar estrutura e identificadores
print("Colunas iguais:", submission.columns.tolist() == sample_submission.columns.tolist())
print("Quantidade de registros igual:", len(submission) == len(sample_submission))
print("Identificadores iguais:", submission["id"].equals(sample_submission["id"]))


Colunas iguais: True
Quantidade de registros igual: True
Identificadores iguais: True


### 4.27. Enviar a submissão ao Kaggle

### 4.28. Conclusão do projeto

O desenvolvimento deste projeto permitiu percorrer as principais etapas de um fluxo de Machine Learning, desde a análise exploratória e preparação dos dados até o treinamento, avaliação e geração de previsões.

Foram desenvolvidos e comparados dois modelos de regressão: Random Forest e CatBoost. Na validação, o CatBoost apresentou o melhor desempenho, com MAE de 1.176,09 e RMSE de 1.852,26.

Com base nessa comparação, o CatBoost foi selecionado para o treinamento final, utilizando os 188.318 registros disponíveis no conjunto de treino. O modelo foi utilizado para gerar previsões para os 125.546 registros do conjunto de teste.

O arquivo de submissão foi validado e enviado ao Kaggle, onde obteve Public Score de 0,27053 e Private Score de 0,60350. A submissão foi registrada após o prazo da competição.

O projeto consolidou a aplicação prática de técnicas de análise de dados, preparação de variáveis categóricas, modelagem preditiva e avaliação de modelos utilizando Python, Pandas, Scikit-learn e CatBoost.

### 4.29. Atualizar o repositório no GitHub